# Feature Engineering — Team 5 SentimenTrip
**Author:** Saloni Jain  
**Stage:** 03 — Feature Engineering  
**Catalog:** `msbabigdata` | **Schema:** `default`  
**Input:** `msbabigdata.default.lda_topic_results` (from Lear's LDA)  
**Output:** `msbabigdata.default.business_profiles` + `msbabigdata.default.user_personas`   

---
## What This Notebook Does
1. Loads Lear's LDA topic assignments from `msbabigdata.default.lda_topic_results`
2. Builds a **business profile vector** — what is each restaurant known for?
3. Adds **VADER sentiment scores** — computed from raw Yelp JSON (no temp view dependency)
4. Adds a **dominant topic vibe tag** — the #1 thing each business is known for
5. Builds a **user profile vector** — what does each reviewer care about?
6. Runs **K-Means clustering** to group users into behavioral personas
7. Computes **persona confidence score** — how focused is each user's taste profile?
8. Saves both tables to `msbabigdata.default` for Quinten's RAG layer

---
This project repository is created in partial fulfillment of the requirements for the Big Data Analytics course offered by the Master of Science in Business Analytics program at the Carlson School of Management, University of Minnesota.


## Cell 1 — Install Dependencies

In [0]:
%pip install nltk vaderSentiment scipy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 792.4/792.4 kB 32.3 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## Cell 2 — Import Libraries

In [0]:
# Core libraries
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, FloatType

import pandas as pd
import numpy as np

# Sentiment analysis
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')

# Entropy for persona confidence
from scipy.stats import entropy as scipy_entropy

print("All libraries imported successfully!")

All libraries imported successfully!


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/spark-b9538251-be42-4a13-8e4e-dc/nltk_data...


## Cell 3 — Load Lear's LDA Output

**What this does:** Loads the Delta table that Lear saved after running LDA on 343K Philadelphia reviews.  
**Schema:** `review_id | business_id | user_id | topic_id | topic_label | topic_prob`  
**Expected rows:** ~4.76M (avg 13.9 topic rows per review across 25 topics)

In [0]:
# Load Lear's LDA topic results from msbabigdata.default catalog
topic_df = spark.table("msbabigdata.default.lda_topic_results")

# Verify it loaded correctly
print("=" * 50)
print("LDA Topic Results loaded!")
print(f"Total rows: {topic_df.count():,}")
print(f"Unique reviews: {topic_df.select('review_id').distinct().count():,}")
print(f"Unique businesses: {topic_df.select('business_id').distinct().count():,}")
print(f"Unique users: {topic_df.select('user_id').distinct().count():,}")
print(f"Unique topics: {topic_df.select('topic_label').distinct().count()}")
print("=" * 50)
topic_df.show(10, truncate=False)


LDA Topic Results loaded!
Total rows: 3,854,659
Unique reviews: 261,571
Unique businesses: 1,962
Unique users: 113,952
Unique topics: 25
+----------------------+----------------------+----------------------+--------+--------------------------+----------+
|review_id             |business_id           |user_id               |topic_id|topic_label               |topic_prob|
+----------------------+----------------------+----------------------+--------+--------------------------+----------+
|rEMd2JZiOVu-nDfb9bLeug|CYSPKiVdoPX3erovujnE9Q|rOf1Y62zS2-YFXAp6Okzzw|22      |Cocktail Bars & Speakeasy |0.0124    |
|rEMd2JZiOVu-nDfb9bLeug|CYSPKiVdoPX3erovujnE9Q|rOf1Y62zS2-YFXAp6Okzzw|23      |Overall Food Quality      |0.0533    |
|ypmfMnEpwI5sCG6P296wSg|-0TffRSXXIlBYVbb5AwfTg|5cBooky8Y5_q5FV2zK-hPg|0       |Service & Wait Time       |0.286     |
|ypmfMnEpwI5sCG6P296wSg|-0TffRSXXIlBYVbb5AwfTg|5cBooky8Y5_q5FV2zK-hPg|1       |Philly Neighborhood Gems  |0.1062    |
|ypmfMnEpwI5sCG6P296wSg|-0TffRSXXIlBY

## Cell 4 — Confirm All 25 Topics Are Present

**Why:** Before building profiles, confirm Lear's 25 topic labels are all present and check which topics appear most frequently.

In [0]:
# Show all 25 topics with review count and average probability
print("All 25 LDA Topics from Lear's model:")
print("=" * 60)

topic_summary = (
    topic_df
    .groupBy("topic_id", "topic_label")
    .agg(
        F.count("review_id").alias("review_count"),
        F.round(F.avg("topic_prob"), 4).alias("avg_prob")
    )
    .orderBy("topic_id")
)

topic_summary.show(25, truncate=False)

All 25 LDA Topics from Lear's model:
+--------+-------------------------------+------------+--------+
|topic_id|topic_label                    |review_count|avg_prob|
+--------+-------------------------------+------------+--------+
|0       |Service & Wait Time            |261571      |0.2704  |
|1       |Philly Neighborhood Gems       |261571      |0.1361  |
|2       |Food Trucks & Trendy Spots     |38342       |0.0267  |
|3       |Unique & Quirky Dining         |56752       |0.0255  |
|4       |Cafe, Coffee & Reading Terminal|110150      |0.0273  |
|5       |Venue & Location Experience    |79645       |0.0251  |
|6       |Value & Portion Size           |261212      |0.0518  |
|7       |Brunch & Breakfast             |44963       |0.0434  |
|8       |Outdoor & Street Seating       |175319      |0.0224  |
|9       |Bar & Nightlife Crowd          |65413       |0.0235  |
|10      |Desserts & Bakery              |158715      |0.0264  |
|11      |Craft Beer & Sports Bars       |134775     

## Cell 5 — Build Business Profiles

**What this does:**  
For every business, aggregate all review topic probabilities.  
Each business becomes a 25-dimensional vector.  

**Example:**  
La Viola restaurant → `{Cocktail Bar & Ambiance: 0.45, Fine Dining: 0.30, Service: 0.25...}`  

**Why this matters:**  
These vectors go into FAISS. When a user types "romantic dinner," FAISS finds the businesses closest to that query in this 25-dimensional space.

In [0]:
# Build business profiles by pivoting topic_label into columns
# Each business gets one row with 25 topic score columns

biz_profiles_spark = (
    topic_df
    .groupBy("business_id", "topic_label")
    .agg(
        F.round(F.avg("topic_prob"), 4).alias("avg_prob"),
        F.count("review_id").alias("mention_count")
    )
    .groupBy("business_id")
    .pivot("topic_label")         # each topic label becomes a column
    .agg(F.avg("avg_prob"))       # one value per business per topic
    .fillna(0)                    # topics not mentioned = 0
)

print(f"Business profiles: {biz_profiles_spark.count():,} businesses")
print(f"Columns: {len(biz_profiles_spark.columns)} (1 business_id + 25 topics)")
biz_profiles_spark.show(3, truncate=True)

Business profiles: 1,962 businesses
Columns: 26 (1 business_id + 25 topics)
+--------------------+---------------------+----------------------+------------------+-------------------------------+-------------------------+------------------------+-------------------------+-------------------------+------------------------+------------------------+-----------------+-------------------+--------------------------+--------------------------+--------------------------+------------------------+--------------------+------------------------+-------------------+-------------------+------------------------+---------------------+----------------------+--------------------+---------------------------+
|         business_id|Bar & Nightlife Crowd|Bar Vibes & Live Music|Brunch & Breakfast|Cafe, Coffee & Reading Terminal|Casual Payment & Ordering|Chef Specials & Platters|Cocktail Bars & Speakeasy|Comfort Food & Sandwiches|Craft Beer & Sports Bars|Customer Service Quality|Desserts & Bakery|Dinner & Happy

## Cell 6 — Add Business Metadata (Name, Category, Stars)

**Why:** The business profile vector needs the business name and category so Quinten can display real names in recommendations, not just IDs.

In [0]:
# Load business metadata from msbabigdata.default.business_metadata
# (saved by Lear's BDA_Project_v3, includes is_open and hours)
biz_meta = spark.table("msbabigdata.default.business_metadata").select(
    "business_id",
    F.col("name").alias("business_name"),
    "address",
    "categories",
    F.col("stars").alias("business_stars"),
    "review_count",
    "is_open",
    "hours_monday", "hours_tuesday", "hours_wednesday",
    "hours_thursday", "hours_friday", "hours_saturday", "hours_sunday"
)

# Join metadata onto profiles
biz_profiles_with_meta = biz_profiles_spark.join(
    biz_meta,
    on="business_id",
    how="left"
)

print(f"Business profiles with metadata: {biz_profiles_with_meta.count():,}")
biz_profiles_with_meta.select(
    "business_id", "business_name", "business_stars", "review_count",
    "categories", "is_open", "hours_friday"
).show(5, truncate=False)


Business profiles with metadata: 1,962
+----------------------+---------------------+--------------+------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+------------+
|business_id           |business_name        |business_stars|review_count|categories                                                                                                                                                    |is_open|hours_friday|
+----------------------+---------------------+--------------+------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+------------+
|j6usntbtSzFyuwx4n2mn1g|Las Bugambilias      |4.0           |381         |Restaurants, Mexican                                                                                                      

## Cell 7 — Add Dominant Topic Vibe Tag

**What this does:**  
Finds the single strongest topic for each business.  

**Why this is important for the demo:**  
Every business card in Streamlit will show a one-line vibe tag like:  
- "Morning Glory Diner" → **Brunch & Breakfast**  
- "Bob & Barbara's" → **Bar, Nightlife & Sports**  
- "Vedge" → **Healthy & Light Options**  

Non-technical evaluators understand this instantly — no explanation needed.

In [0]:
biz_pd = biz_profiles_with_meta.toPandas()

metadata_cols = [
    "business_id", "business_name", "address",
    "categories", "business_stars", "review_count",
    "is_open",
    "hours_monday", "hours_tuesday", "hours_wednesday",
    "hours_thursday", "hours_friday", "hours_saturday", "hours_sunday"
]
topic_cols = [c for c in biz_pd.columns if c not in metadata_cols]

# Normalize by dividing each topic score by its average across all businesses
# This makes specific topics stand out over generic ones that apply everywhere
topic_means = biz_pd[topic_cols].mean()
biz_pd_normalized = biz_pd[topic_cols].div(topic_means)

biz_pd["dominant_topic"]       = biz_pd_normalized.idxmax(axis=1)
biz_pd["dominant_topic_score"] = biz_pd[topic_cols].max(axis=1).round(4)
biz_pd["topic_diversity"]      = (biz_pd[topic_cols] > 0.05).sum(axis=1)

print(f"Topic columns: {len(topic_cols)}")
print("Sample business vibe tags:")
print(biz_pd[["business_name", "dominant_topic", "dominant_topic_score", "topic_diversity"]]
      .dropna(subset=["business_name"])
      .head(15)
      .to_string(index=False))

Topic columns: 25
Sample business vibe tags:
            business_name                  dominant_topic  dominant_topic_score  topic_diversity
          Las Bugambilias        Small Plates & Cocktails                0.2622                5
                    Tango          Bar Vibes & Live Music                0.3012                4
      Kensington Quarters      Fine Dining & Tasting Menu                0.2459                5
    Libertine Restauraunt              Brunch & Breakfast                0.2577                5
               McDonald's        Customer Service Quality                0.3567                5
              Hale & True        Craft Beer & Sports Bars                0.2365                5
               El Merkury        Small Plates & Cocktails                0.2692                6
             PrimoHoagies       Casual Payment & Ordering                0.3306                5
                   Target      Markets & Grocery Shopping                0.3180   

## Cell 8 — Add VADER Sentiment Scores Per Business Per Topic

**What this does:**  
For each business, for each topic, computes whether reviews mentioning that topic are POSITIVE or NEGATIVE.  

**Why this is the feature that makes you win:**  
Two restaurants can both have high "Service & Wait Time" probability.  
Without sentiment: they look identical.  
With sentiment: Restaurant A = +0.8 (people rave), Restaurant B = -0.6 (people complain).  
Now you know which one to recommend.  

**VADER score range:** -1.0 (very negative) to +1.0 (very positive), 0 = neutral.

In [0]:
# Load review text for VADER — reads from raw Yelp JSON
# (no temp view dependency — works in any session)
print("Loading review text for VADER...")

BASE_PATH = "/Volumes/msbabigdata/trends/trendsmarket/yelp_dataset/extracted/"

reviews_text_spark = (
    spark.read.json(BASE_PATH + "yelp_academic_dataset_review.json")
    .select("review_id", "business_id", "text", "stars")
    .withColumnRenamed("text", "review_text")
    .withColumnRenamed("stars", "review_stars")
    .join(
        spark.table("msbabigdata.default.lda_topic_results")
            .select("review_id").distinct(),
        on="review_id",
        how="inner"  # only Philly food reviews that are in LDA table
    )
)

# Convert to pandas for VADER (VADER runs on single node, not distributed)
reviews_pd = reviews_text_spark.toPandas()
print(f"Reviews loaded: {len(reviews_pd):,}")

# Compute VADER compound sentiment score for each review
# compound: -1 = most negative, +1 = most positive, 0 = neutral
print("Computing VADER sentiment scores... (~5 minutes)")
sid = SentimentIntensityAnalyzer()

reviews_pd["sentiment"] = reviews_pd["review_text"].apply(
    lambda x: round(sid.polarity_scores(str(x))["compound"], 4)
)

print(f"Sentiment computed! Score range: {reviews_pd['sentiment'].min():.2f} to {reviews_pd['sentiment'].max():.2f}")
print(f"Average sentiment: {reviews_pd['sentiment'].mean():.3f}")
reviews_pd[["review_id", "sentiment"]].head()


Loading review text for VADER...
Reviews loaded: 261,571
Computing VADER sentiment scores... (~5 minutes)
Sentiment computed! Score range: -1.00 to 1.00
Average sentiment: 0.715


,review_id,sentiment
0,Xs8Z8lmKkosqW5mw_sVAoA,0.9785
1,JBWZmBy69VMggxj3eYn17Q,0.9168
2,YcLXh-3UC9y6YFAI9xxzPQ,0.9318
3,DWbmJF84jRrGaJRmlSSnYQ,0.9363
4,-7LkjSPzfVgnVpuVuRuOow,0.9730


In [0]:
# Convert sentiment back to Spark and join with topic assignments
sentiment_spark = spark.createDataFrame(
    reviews_pd[["review_id", "sentiment"]]
)

# Join: each topic row now also has the review's sentiment score
topic_with_sent = topic_df.join(
    sentiment_spark,
    on="review_id",
    how="left"
)

# For each business + topic combination, compute:
# - avg_topic_prob: how much is this topic discussed?
# - avg_sentiment: when it IS discussed, is it positive or negative?
biz_sentiment = (
    topic_with_sent
    .groupBy("business_id", "topic_label")
    .agg(
        F.round(F.avg("topic_prob"), 4).alias("topic_prob"),
        F.round(F.avg("sentiment"), 4).alias("topic_sentiment")
    )
)

print("Sample sentiment per business per topic:")
biz_sentiment.show(10, truncate=False)

Sample sentiment per business per topic:
+----------------------+-------------------------------+----------+---------------+
|business_id           |topic_label                    |topic_prob|topic_sentiment|
+----------------------+-------------------------------+----------+---------------+
|73UjNbSoQjQAOS45rcihFg|Spicy & Asian Flavors          |0.023     |0.7964         |
|73UjNbSoQjQAOS45rcihFg|Small Plates & Cocktails       |0.065     |0.8025         |
|S8ZFYEgMejpChID8tzKo9A|Dinner & Happy Hour            |0.1235    |0.8193         |
|iUZEGx29miZObLd6_lt7Vg|Cafe, Coffee & Reading Terminal|0.0238    |0.7768         |
|_f3JQU6IXpGmTLaSqGy79g|Craft Beer & Sports Bars       |0.0645    |0.751          |
|nF5QByCz21MY0KLtSoB5KA|Fine Dining & Tasting Menu     |0.0278    |0.7333         |
|3c3okC0udBKZD_fASFrI-A|Small Plates & Cocktails       |0.0133    |0.7087         |
|3c3okC0udBKZD_fASFrI-A|Fine Dining & Tasting Menu     |0.0199    |0.6748         |
|EzjysPg2-lVX1E0ibStUXw|Markets & G

In [0]:
# Add overall sentiment score to business pandas dataframe
# This is a single overall score per business (avg across all reviews)
overall_sentiment = reviews_pd.groupby("business_id")["sentiment"].mean().reset_index()
overall_sentiment.columns = ["business_id", "overall_sentiment"]
overall_sentiment["overall_sentiment"] = overall_sentiment["overall_sentiment"].round(4)

# Merge into business profiles
biz_pd = biz_pd.merge(overall_sentiment, on="business_id", how="left")

print("Top 10 most positively reviewed businesses:")
print(
    biz_pd[["business_name", "overall_sentiment", "dominant_topic", "business_stars"]]
    .dropna()
    .sort_values("overall_sentiment", ascending=False)
    .head(10)
    .to_string(index=False)
)

Top 10 most positively reviewed businesses:
                                   business_name  overall_sentiment                  dominant_topic  business_stars
                                            Cook             0.9693      Fine Dining & Tasting Menu             4.5
                                Restaurant Ambra             0.9453      Fine Dining & Tasting Menu             5.0
The Restaurant at JNA Institute of Culinary Arts             0.9416      Fine Dining & Tasting Menu             4.0
                            Miss Rachel's Pantry             0.9401 Cafe, Coffee & Reading Terminal             5.0
                       Balkan Express Restaurant             0.9378           Spicy & Asian Flavors             4.5
                                 The Library Bar             0.9364          Bar Vibes & Live Music             4.0
                                 Ken Love's BYOB             0.9353      Fine Dining & Tasting Menu             4.5
                            

## Cell 9 — Build User Profiles

**What this does:**  
For every user, aggregate their review topic probabilities.  
Each user becomes a 25-dimensional vector representing their taste preferences.  

**Example:**  
User who always reviews coffee shops:  
`{Cafe & Coffee: 0.52, Healthy & Light Options: 0.31, Service & Wait Time: 0.24...}`  

**Filter:** Only users with ≥3 reviews — need enough signal to build a meaningful profile.

In [0]:
# Count reviews per user first
user_review_counts = (
    topic_df
    .groupBy("user_id")
    .agg(F.countDistinct("review_id").alias("review_count"))
)

print(f"Total unique users: {user_review_counts.count():,}")

# Show review count distribution
user_review_counts.groupBy("review_count").count().orderBy("review_count").show(10)

# Filter to users with at least 3 reviews
active_users = user_review_counts.filter(F.col("review_count") >= 3)
print(f"Users with 3+ reviews: {active_users.count():,}")

Total unique users: 113,952
+------------+-----+
|review_count|count|
+------------+-----+
|           1|76689|
|           2|17246|
|           3| 6994|
|           4| 3738|
|           5| 2148|
|           6| 1393|
|           7|  955|
|           8|  718|
|           9|  589|
|          10|  402|
+------------+-----+
only showing top 10 rows
Users with 3+ reviews: 20,017


In [0]:
# Build user profiles — same pivot logic as business profiles
user_profiles_spark = (
    topic_df
    .join(active_users.select("user_id"), on="user_id", how="inner")
    .groupBy("user_id", "topic_label")
    .agg(F.round(F.avg("topic_prob"), 4).alias("avg_prob"))
    .groupBy("user_id")
    .pivot("topic_label")
    .agg(F.avg("avg_prob"))
    .fillna(0)
)

# Add review count back
user_profiles_spark = user_profiles_spark.join(
    active_users,
    on="user_id",
    how="left"
)

print(f"User profiles built: {user_profiles_spark.count():,} users")
print(f"Columns: {len(user_profiles_spark.columns)}")
user_profiles_spark.show(3)

User profiles built: 20,017 users
Columns: 27
+--------------------+---------------------+----------------------+------------------+-------------------------------+-------------------------+------------------------+-------------------------+-------------------------+------------------------+------------------------+-----------------+-------------------+--------------------------+--------------------------+--------------------------+------------------------+--------------------+------------------------+-------------------+-------------------+------------------------+---------------------+----------------------+--------------------+---------------------------+------------+
|             user_id|Bar & Nightlife Crowd|Bar Vibes & Live Music|Brunch & Breakfast|Cafe, Coffee & Reading Terminal|Casual Payment & Ordering|Chef Specials & Platters|Cocktail Bars & Speakeasy|Comfort Food & Sandwiches|Craft Beer & Sports Bars|Customer Service Quality|Desserts & Bakery|Dinner & Happy Hour|Fine Dining

## Cell 11 — User Taste Profile

**Clustering removed** — K-Means is not used in the recommender pipeline.  
Q's RAG uses the raw 25-dim topic vector directly via cosine similarity.  
Instead of a cluster label, each user gets their top 3 topic names for Streamlit display.


In [0]:
# No clustering — derive taste profile from top 3 topics
# Uses normalization so generic topics dont dominate

cluster_feature_cols = [
    c for c in user_profiles_spark.columns
    if c not in ["user_id", "review_count"]
]

# Convert to pandas
users_pd = user_profiles_spark.toPandas()

# Normalize — divide each topic score by its average across all users
# This stops generic topics like Service & Wait Time dominating
topic_means_user = users_pd[cluster_feature_cols].mean()
users_pd_normalized = users_pd[cluster_feature_cols].div(topic_means_user)

users_pd["top_taste_1"] = users_pd_normalized.apply(lambda row: row.nlargest(1).index[0], axis=1)
users_pd["top_taste_2"] = users_pd_normalized.apply(lambda row: row.nlargest(2).index[1], axis=1)
users_pd["top_taste_3"] = users_pd_normalized.apply(lambda row: row.nlargest(3).index[2], axis=1)
users_pd["persona_label"] = users_pd["top_taste_1"]

print(f"User profiles ready: {len(users_pd):,} users")
print("Top taste distribution after normalization:")
print(users_pd["top_taste_1"].value_counts().head(10))


User profiles ready: 20,017 users
Top taste distribution after normalization:
Brunch & Breakfast                 2172
Food Trucks & Trendy Spots         2010
Cocktail Bars & Speakeasy          1925
Chef Specials & Platters           1572
Unique & Quirky Dining             1222
Cafe, Coffee & Reading Terminal    1097
Small Plates & Cocktails           1059
Desserts & Bakery                  1038
Craft Beer & Sports Bars            990
Bar & Nightlife Crowd               980
Name: top_taste_1, dtype: int64


## Cell 12 — Persona Confidence Score

**What this does:**  
Measures how "focused" each user's taste profile is.  

**How:** Uses entropy. Low entropy = concentrated on few topics = high confidence.  
High entropy = spread across all 25 topics = mixed/unclear persona.  

**Why this is great for the demo:**  
The Streamlit persona card shows:  
"Your taste profile: **Brunch Lover** (confidence: 87%)"  
People at the booth immediately get it — it's like a personality test result.

In [0]:
# Compute persona confidence score
# users_pd already in pandas from Cell 11 — no toPandas needed here

def compute_confidence(row):
    probs = np.array([row[c] for c in cluster_feature_cols], dtype=float)
    probs = np.clip(probs, 1e-10, None)
    probs = probs / probs.sum()
    max_entropy = np.log(len(cluster_feature_cols))
    actual_entropy = scipy_entropy(probs)
    confidence = 1 - (actual_entropy / max_entropy)
    return round(float(confidence), 3)

users_pd["persona_confidence"]     = users_pd.apply(compute_confidence, axis=1)
users_pd["persona_confidence_pct"] = (users_pd["persona_confidence"] * 100).round(1)

print("Persona confidence score distribution:")
print(users_pd["persona_confidence_pct"].describe())
print("Sample users with confidence scores:")
print(
    users_pd[["user_id", "persona_label", "persona_confidence_pct", "review_count"]]
    .sort_values("persona_confidence_pct", ascending=False)
    .head(10)
    .to_string(index=False)
)


Persona confidence score distribution:
count    20017.000000
mean        17.984873
std          5.178346
min          6.100000
25%         14.000000
50%         18.000000
75%         21.600000
max         48.400000
Name: persona_confidence_pct, dtype: float64
Sample users with confidence scores:
               user_id             persona_label  persona_confidence_pct  review_count
FaBY9v5YRnCNUhKSJWBb_g       Service & Wait Time                    48.4             3
YVMs5ZPS3LmYYBVZmA66IQ  Outdoor & Street Seating                    42.4             3
KQirXqhe8eKKX3iX8uKzkg  Customer Service Quality                    41.2             3
hAGKvFCQXXEQcM8ljgXJQQ Casual Payment & Ordering                    40.4             3
VKWWsh-QQzdGJx4xVcio0g       Service & Wait Time                    40.4             3
qqNKErWW4M-As9KnMtyg6A  Chef Specials & Platters                    40.1             3
oRCYGVVt0yjTd-vvTyChhA     Bar & Nightlife Crowd                    39.8             3
Ux9jkpG

## Cell 13 — Add Intent Combo Scores (Bonus Feature)

**What this does:**  
Creates composite scores for specific dining intents by combining multiple topics.  
These are the features that power "intent-based" recommendations.  

**Examples:**  
- `romantic_score` = Fine Dining × 0.4 + Cocktail Bar × 0.4 + Outdoor Seating × 0.2  
- `solo_work_score` = Cafe & Coffee × 0.5 + Healthy Options × 0.3 + (1 - Nightlife) × 0.2  
- `family_score` = Brunch × 0.3 + Overall Value × 0.3 + Healthy × 0.2 + (1 - Nightlife) × 0.2  

When a user types "romantic dinner", the system finds businesses with highest `romantic_score`.

In [0]:
# Intent scores — computed using actual topic column names from LDA
# Column names match what LDA produced for this run

# Romantic / Date Night
biz_pd["romantic_score"] = (
    biz_pd.get("Fine Dining & Tasting Menu", pd.Series([0]*len(biz_pd))) * 0.40 +
    biz_pd.get("Cocktail Bars & Speakeasy", pd.Series([0]*len(biz_pd))) * 0.35 +
    biz_pd.get("Outdoor & Street Seating", pd.Series([0]*len(biz_pd))) * 0.25
).round(4)

# Solo Work / Coffee
biz_pd["solo_work_score"] = (
    biz_pd.get("Cafe, Coffee & Reading Terminal", pd.Series([0]*len(biz_pd))) * 0.50 +
    biz_pd.get("Quick & Fresh Lunch", pd.Series([0]*len(biz_pd))) * 0.30 +
    (1 - biz_pd.get("Bar & Nightlife Crowd", pd.Series([0]*len(biz_pd)))) * 0.20
).round(4)

# Family
biz_pd["family_score"] = (
    biz_pd.get("Brunch & Breakfast", pd.Series([0]*len(biz_pd))) * 0.30 +
    biz_pd.get("Value & Portion Size", pd.Series([0]*len(biz_pd))) * 0.30 +
    biz_pd.get("Outdoor & Street Seating", pd.Series([0]*len(biz_pd))) * 0.20 +
    (1 - biz_pd.get("Cocktail Bars & Speakeasy", pd.Series([0]*len(biz_pd)))) * 0.20
).round(4)

# Group Celebration
biz_pd["group_celebration_score"] = (
    biz_pd.get("Bar & Nightlife Crowd", pd.Series([0]*len(biz_pd))) * 0.35 +
    biz_pd.get("Cocktail Bars & Speakeasy", pd.Series([0]*len(biz_pd))) * 0.35 +
    biz_pd.get("Bar Vibes & Live Music", pd.Series([0]*len(biz_pd))) * 0.30
).round(4)

# Hidden Gem
biz_pd["hidden_gem_score"] = (
    biz_pd.get("Philly Neighborhood Gems", pd.Series([0]*len(biz_pd))) * 0.40 +
    biz_pd.get("Unique & Quirky Dining", pd.Series([0]*len(biz_pd))) * 0.30 +
    biz_pd.get("Value & Portion Size", pd.Series([0]*len(biz_pd))) * 0.30
).round(4)

print('Intent scores computed:')
for col in ['romantic_score','solo_work_score','family_score','group_celebration_score','hidden_gem_score']:
    print(f'  {col}: mean={biz_pd[col].mean():.4f}, zeros={(biz_pd[col]==0).sum()}')

print('Top 5 ROMANTIC restaurants:')
print(biz_pd[["business_name","romantic_score","dominant_topic"]]
      .dropna(subset=["business_name"])
      .sort_values("romantic_score",ascending=False)
      .head(5).to_string(index=False))

print('Top 5 GROUP spots:')
print(biz_pd[["business_name","group_celebration_score","dominant_topic"]]
      .dropna(subset=["business_name"])
      .sort_values("group_celebration_score",ascending=False)
      .head(5).to_string(index=False))


Intent scores computed:
  romantic_score: mean=0.0261, zeros=0
  solo_work_score: mean=0.2225, zeros=0
  family_score: mean=0.2240, zeros=0
  group_celebration_score: mean=0.0276, zeros=0
  hidden_gem_score: mean=0.0776, zeros=0
Top 5 ROMANTIC restaurants:
       business_name  romantic_score             dominant_topic
              Laurel          0.0539 Fine Dining & Tasting Menu
Felly Bistro On Pass          0.0537  Cocktail Bars & Speakeasy
                Cook          0.0537 Fine Dining & Tasting Menu
    Restaurant Ambra          0.0525 Fine Dining & Tasting Menu
         River Twice          0.0494 Fine Dining & Tasting Menu
Top 5 GROUP spots:
                business_name  group_celebration_score         dominant_topic
              Tavern On Camac                   0.0664 Bar Vibes & Live Music
           The Dolphin Tavern                   0.0663 Bar Vibes & Live Music
       First Unitarian Church                   0.0655 Bar Vibes & Live Music
Howl At the Moon Philadelphi

## Cell 14 — Save Business Profiles to Shared Catalog

**This is your handoff to Quinten.**  
Once this cell runs, Quinten can immediately start building the FAISS index.

In [0]:
import re
def clean_col(c):
    c = re.sub(r'[^a-zA-Z0-9_]', '_', c)
    c = re.sub(r'_+', '_', c)  # collapse multiple underscores
    return c.strip('_').lower()

biz_pd.columns = [clean_col(c) for c in biz_pd.columns]

biz_final_spark = spark.createDataFrame(biz_pd)

print("Saving business profiles to msbabigdata.default...")
biz_final_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("msbabigdata.default.business_profiles")

verify_biz = spark.table("msbabigdata.default.business_profiles")
print("=" * 50)
print("Business profiles saved!")
print(f"Rows: {verify_biz.count():,}")
print(f"Columns: {len(verify_biz.columns)}")
key_cols = ["business_id","business_name","dominant_topic",
            "overall_sentiment","romantic_score","solo_work_score",
            "family_score","group_celebration_score","hidden_gem_score",
            "is_open","hours_friday"]
print("Key columns:", [c for c in key_cols if c in verify_biz.columns])
print("=" * 50)


Saving business profiles to msbabigdata.default...
Business profiles saved!
Rows: 1,962
Columns: 48
Key columns: ['business_id', 'business_name', 'dominant_topic', 'overall_sentiment', 'romantic_score', 'solo_work_score', 'family_score', 'group_celebration_score', 'hidden_gem_score', 'is_open', 'hours_friday']


## Cell 15 — Save User Personas to Shared Catalog

In [0]:
import re
def clean_col(c):
    c = re.sub(r'[^a-zA-Z0-9_]', '_', c)
    c = re.sub(r'_+', '_', c)  # collapse multiple underscores
    return c.strip('_').lower()

users_pd.columns = [clean_col(c) for c in users_pd.columns]

user_final_spark = spark.createDataFrame(users_pd)

print("Saving user personas to msbabigdata.default...")
user_final_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("msbabigdata.default.user_personas")

verify_users = spark.table("msbabigdata.default.user_personas")
print("=" * 50)
print("User personas saved!")
print(f"Rows: {verify_users.count():,}")
print("Top taste distribution:")
verify_users.groupBy("top_taste_1").count().orderBy("count",ascending=False).show(10)
print("=" * 50)


Saving user personas to msbabigdata.default...
User personas saved!
Rows: 20,017
Top taste distribution:
+--------------------+-----+
|         top_taste_1|count|
+--------------------+-----+
|  Brunch & Breakfast| 2172|
|Food Trucks & Tre...| 2010|
|Cocktail Bars & S...| 1925|
|Chef Specials & P...| 1572|
|Unique & Quirky D...| 1222|
|Cafe, Coffee & Re...| 1097|
|Small Plates & Co...| 1059|
|   Desserts & Bakery| 1038|
|Craft Beer & Spor...|  990|
|Bar & Nightlife C...|  980|
+--------------------+-----+
only showing top 10 rows


## Cell 16 — Final Summary + Handoff Message

Run this cell last to get a summary of everything produced.

In [0]:
print("=" * 60)
print("FEATURE ENGINEERING COMPLETE — SALONI")
print("=" * 60)

biz_count  = spark.table("msbabigdata.default.business_profiles").count()
user_count = spark.table("msbabigdata.default.user_personas").count()

print(f"Business Profiles")
print(f"   Table: msbabigdata.default.business_profiles")
print(f"   Rows : {biz_count:,}")
print(f"   Features: 25 topic vectors + sentiment + vibe tags + intent scores + is_open + hours")

print(f"User Personas")
print(f"   Table: msbabigdata.default.user_personas")
print(f"   Rows : {user_count:,}")
print(f"   Features: 25 topic vectors + top_taste_1/2/3 + persona_confidence")

print(f"Intent Scores:")
print("   romantic_score, solo_work_score, family_score,")
print("   group_celebration_score, hidden_gem_score")

print("Quinten — tables ready:")
print(spark.table("msbabigdata.default.business_profiles"))
print(spark.table("msbabigdata.default.user_personas"))


FEATURE ENGINEERING COMPLETE — SALONI
Business Profiles
   Table: msbabigdata.default.business_profiles
   Rows : 1,962
   Features: 25 topic vectors + sentiment + vibe tags + intent scores + is_open + hours
User Personas
   Table: msbabigdata.default.user_personas
   Rows : 20,017
   Features: 25 topic vectors + top_taste_1/2/3 + persona_confidence
Intent Scores:
   romantic_score, solo_work_score, family_score,
   group_celebration_score, hidden_gem_score
Quinten — tables ready:
DataFrame[business_id: string, bar_nightlife_crowd: double, bar_vibes_live_music: double, brunch_breakfast: double, cafe_coffee_reading_terminal: double, casual_payment_ordering: double, chef_specials_platters: double, cocktail_bars_speakeasy: double, comfort_food_sandwiches: double, craft_beer_sports_bars: double, customer_service_quality: double, desserts_bakery: double, dinner_happy_hour: double, fine_dining_tasting_menu: double, food_trucks_trendy_spots: double, markets_grocery_shopping: double, outdoor_s

In [0]:
# Export business profiles to Volume
spark.table("msbabigdata.default.business_profiles") \
    .coalesce(1) \
    .write.option("header", "true") \
    .mode("overwrite") \
    .csv("/Volumes/msbabigdata/trends/trendsmarket/business_profiles_export")

# Export user personas to Volume
spark.table("msbabigdata.default.user_personas") \
    .coalesce(1) \
    .write.option("header", "true") \
    .mode("overwrite") \
    .csv("/Volumes/msbabigdata/trends/trendsmarket/user_personas_export")

print("Exported!")

Exported!


In [0]:
# Load both saved tables
biz = spark.table("msbabigdata.default.business_profiles").toPandas()
users = spark.table("msbabigdata.default.user_personas").toPandas()

print("=" * 50)
print("BUSINESS PROFILES VALIDATION")
print("=" * 50)
print(f"Total businesses: {len(biz):,}")
print(f"Total columns: {len(biz.columns)}")

# Check dominant topics are varied
print("\nDominant topic distribution (should be varied, not all same):")
print(biz["dominant_topic"].value_counts().head(10))

# Check intent scores are not all zero
print("\nIntent score averages (should be non-zero):")
intent_cols = ["intent_romantic", "intent_solo_work", "intent_family", 
               "intent_group", "intent_hidden_gem"]
for col in intent_cols:
    if col in biz.columns:
        print(f"  {col}: avg={biz[col].mean():.4f}, zeros={( biz[col]==0).sum()}")

# Check sentiment exists
print(f"\nSentiment avg: {biz['overall_sentiment'].mean():.3f}")
print(f"Sentiment nulls: {biz['overall_sentiment'].isna().sum()}")

print("\n" + "=" * 50)
print("USER PERSONAS VALIDATION")
print("=" * 50)
print(f"Total users: {len(users):,}")
print(f"Total columns: {len(users.columns)}")
print(f"\nTop taste distribution:")
print(users["top_taste_1"].value_counts().head(10))
print(f"\nConfidence score avg: {users['persona_confidence_pct'].mean():.1f}%")

BUSINESS PROFILES VALIDATION
Total businesses: 1,962
Total columns: 48

Dominant topic distribution (should be varied, not all same):
Spicy & Asian Flavors              246
Brunch & Breakfast                 180
Comfort Food & Sandwiches          158
Bar Vibes & Live Music             158
Cafe, Coffee & Reading Terminal    155
Craft Beer & Sports Bars           155
Small Plates & Cocktails           133
Fine Dining & Tasting Menu         113
Desserts & Bakery                  112
Quick & Fresh Lunch                 88
Name: dominant_topic, dtype: int64

Intent score averages (should be non-zero):

Sentiment avg: 0.670
Sentiment nulls: 0

USER PERSONAS VALIDATION
Total users: 20,017
Total columns: 33

Top taste distribution:
Brunch & Breakfast                 2172
Food Trucks & Trendy Spots         2010
Cocktail Bars & Speakeasy          1925
Chef Specials & Platters           1572
Unique & Quirky Dining             1222
Cafe, Coffee & Reading Terminal    1097
Small Plates & Cocktails  